# Unificación del esquema de depth chart (2024 vs. 2025+)

## Qué responde este notebook

Desde Fase 2 se sabe que `nflreadpy` regresa un esquema de depth chart distinto para
temporadas ≤2024 que para 2025 en adelante, sin columna de temporada/semana compartida
(ver `docs/HALLAZGOS.md`). Fase 5 heredó esta limitación sin resolverla: `depth_team`
-- la **segunda variable más importante** del modelo en recepciones/yardas
(`11_seleccion_features.ipynb`) -- sale **100% nulo para 2025 en adelante**, documentado
como limitación conocida en los 3 modelos guardados (`weekly/wr/models/*_metadata.json`).

Es la mejora de mayor prioridad identificada al cierre de Fase 5: el dato ya existe en
`nflreadpy`, solo no se estaba aprovechando por una limitación técnica de mapeo, no por
falta de fuente. Este notebook:

1. Muestra el problema real (el esquema nuevo no tiene semana, solo una fecha de scrape).
2. Explica y verifica la lógica de unificación con un caso concreto antes de aplicarla a
   todo el histórico.
3. Mide la cobertura real de `depth_team` antes y después.
4. Verifica con casos reales conocidos (Marvin Harrison Jr., Ja'Marr Chase) que el
   mapeo produce alineaciones que coinciden con la realidad.
5. Cuantifica el impacto real en las métricas de los 3 modelos ya guardados -- **con los
   mismos hiperparámetros ya elegidos, sin buscar nada nuevo** (este notebook es barato
   a propósito: nada de búsqueda de hiperparámetros ni entrenamientos pesados).

La función resultante (`data.cargar_depth_charts_unificado`) ya está integrada en
`features.construir_tabla_modelado` -- los notebooks 12-20 seguirán funcionando igual
para temporadas ≤2024 (esa parte no cambió) y automáticamente obtendrán `depth_team` real
para 2025+ si se vuelven a ejecutar.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import matplotlib.pyplot as plt
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"


## 1. El problema real: dos esquemas, uno sin semana

`cargar_depth_charts_historico()` (≤2024) trae `season`/`week` directos. El esquema nuevo
(2025+, vía `nfl.load_depth_charts()` crudo) no -- solo trae `dt`, la fecha/hora exacta del
scrape. Sin `season`/`week`, no hay forma directa de cruzarlo contra `cargar_stats_semanales()`
por (season, week, team, player_id), que es como se integra hoy en `construir_tabla_modelado`.

In [2]:
import nflreadpy as nfl

crudo_2025 = nfl.load_depth_charts([2025]).to_pandas()
print("columnas del esquema 2025+:", list(crudo_2025.columns))
print("¿tiene season o week?", any(c in crudo_2025.columns for c in ("season", "week")))
print(f"\nvalores unicos de dt en 2025+: {crudo_2025['dt'].nunique()} (uno por dia, aproximadamente)")


columnas del esquema 2025+: ['dt', 'team', 'player_name', 'espn_id', 'gsis_id', 'pos_grp_id', 'pos_grp', 'pos_id', 'pos_name', 'pos_abb', 'pos_slot', 'pos_rank']
¿tiene season o week? False

valores unicos de dt en 2025+: 221 (uno por dia, aproximadamente)


## 2. La idea de la solución, con un caso concreto primero

Un depth chart scrapeado un día cualquiera describe la alineación que el equipo está
preparando para su **próximo partido**, no el que ya jugó. Con el calendario real (que ya
incluye partidos programados, jugados o no), se puede mapear cada `dt` al primer partido de
ese equipo en o después de esa fecha.

Ejemplo concreto: Arizona Cardinals (`ARI`), 2025. Su semana 8 fue descanso (bye) -- el
calendario salta de la semana 7 (19 de octubre) a la semana 9 (3 de noviembre).

In [3]:
cal_ari = data.cargar_calendario([2025])
cal_ari = cal_ari[(cal_ari["home_team"] == "ARI") | (cal_ari["away_team"] == "ARI")]
cal_ari = cal_ari[["season", "week", "gameday", "home_team", "away_team"]].sort_values("week")
print("calendario real de ARI, 2025 (nótese el salto semana 7 -> semana 9, bye en semana 8):")
print(cal_ari.to_string(index=False))

dt_ari = crudo_2025[(crudo_2025["team"] == "ARI") & (crudo_2025["pos_abb"] == "WR")]["dt"]
dt_ari = pd.to_datetime(dt_ari).dt.tz_localize(None).drop_duplicates().sort_values()
en_ventana_bye = dt_ari[(dt_ari > "2025-10-19") & (dt_ari <= "2025-11-03")]
print(f"\nscrapes de ARI entre el partido de semana 7 (19-oct) y el de semana 9 (3-nov): {len(en_ventana_bye)}")
print("todos esos scrapes -- pese a caer durante el bye -- describen la alineacion que se prepara")
print("para el SIGUIENTE partido real (semana 9), no una 'semana 8' que no existio para este equipo.")


calendario real de ARI, 2025 (nótese el salto semana 7 -> semana 9, bye en semana 8):
 season  week    gameday home_team away_team
   2025     1 2025-09-07        NO       ARI
   2025     2 2025-09-14       ARI       CAR
   2025     3 2025-09-21        SF       ARI
   2025     4 2025-09-25       ARI       SEA
   2025     5 2025-10-05       ARI       TEN
   2025     6 2025-10-12       IND       ARI
   2025     7 2025-10-19       ARI        GB
   2025     9 2025-11-03       DAL       ARI
   2025    10 2025-11-09       SEA       ARI
   2025    11 2025-11-16       ARI        SF
   2025    12 2025-11-23       ARI       JAX
   2025    13 2025-11-30        TB       ARI
   2025    14 2025-12-07       ARI        LA
   2025    15 2025-12-14       HOU       ARI
   2025    16 2025-12-21       ARI       ATL
   2025    17 2025-12-28       CIN       ARI
   2025    18 2026-01-04        LA       ARI

scrapes de ARI entre el partido de semana 7 (19-oct) y el de semana 9 (3-nov): 14
todos esos scrapes --

Con esa lógica (`pd.merge_asof(..., direction="forward")` de cada `dt` de un equipo contra
las fechas de sus propios partidos), cada scrape del bye se asigna correctamente a la semana 9,
no se pierde ni se inventa una semana 8 inexistente. De todos los `dt` que caen en la misma
ventana equipo-semana se usa solo el más reciente (el más cercano al partido es el más
confiable/final) -- y `pos_rank` (ranking limpio, sin empates, del esquema nuevo) hace el mismo
papel que `depth_team` en el esquema viejo.

Esta lógica ya está implementada en `data.cargar_depth_charts_unificado()` -- combina
`cargar_depth_charts_historico()` (≤2024, sin tocar) con el mapeo anterior (2025+) en un solo
DataFrame `(season, week, team, player_id, depth_team)`.

In [4]:
depth_antes = data.cargar_depth_charts_historico([2024])  # esquema viejo, referencia sin cambios
depth_unificado = data.cargar_depth_charts_unificado(range(2016, 2026))
print("shape unificado (2016-2025):", depth_unificado.shape)
print(depth_unificado.groupby("season")["week"].agg(["min", "max", "nunique"]))

duplicados = depth_unificado.duplicated(subset=["season", "week", "team", "player_id"], keep=False)
print(f"\nfilas duplicadas jugador-equipo-semana: {duplicados.sum()} de {len(depth_unificado)}")


shape unificado (2016-2025): (31636, 5)
        min   max  nunique
season                    
2016.0  1.0  18.0       18
2017.0  1.0  18.0       18
2018.0  1.0  18.0       18
2019.0  1.0  18.0       18
2020.0  1.0  18.0       18
2021.0  1.0  18.0       18
2022.0  1.0  18.0       18
2023.0  1.0  18.0       18
2024.0  1.0  18.0       18
2025.0  1.0  18.0       18

filas duplicadas jugador-equipo-semana: 0 de 31636


## 3. Verificación con casos reales conocidos

Antes de medir cualquier impacto en el modelo, se confirma que el mapeo produce alineaciones que
coinciden con lo que se sabe pasó en realidad: Marvin Harrison Jr. fue WR1 titular de Arizona
casi toda la temporada 2025; Ja'Marr Chase fue WR1 titular de Cincinnati toda la temporada.

In [5]:
jugadores = data.cargar_jugadores()

def depth_team_jugador(nombre_id, equipo_esperado):
    fila = depth_unificado[(depth_unificado["player_id"] == nombre_id) & (depth_unificado["season"] == 2025)]
    return fila.sort_values("week")[["season", "week", "team", "depth_team"]]

mhj_id = jugadores[jugadores["display_name"] == "Marvin Harrison Jr."]["gsis_id"].iloc[0]
chase_id = jugadores[jugadores["display_name"] == "Ja'Marr Chase"]["gsis_id"].iloc[0]

print("Marvin Harrison Jr. (ARI), depth_team por semana 2025:")
print(depth_team_jugador(mhj_id, "ARI").to_string(index=False))

print("\nJa'Marr Chase (CIN), depth_team por semana 2025:")
print(depth_team_jugador(chase_id, "CIN").to_string(index=False))


Marvin Harrison Jr. (ARI), depth_team por semana 2025:
 season  week team  depth_team
 2025.0   1.0  ARI           1
 2025.0   2.0  ARI           1
 2025.0   3.0  ARI           1
 2025.0   4.0  ARI           1
 2025.0   5.0  ARI           1
 2025.0   6.0  ARI           1
 2025.0   7.0  ARI           1
 2025.0   9.0  ARI           1
 2025.0  10.0  ARI           1
 2025.0  11.0  ARI           1
 2025.0  12.0  ARI           1
 2025.0  13.0  ARI           1
 2025.0  14.0  ARI           1
 2025.0  15.0  ARI           1
 2025.0  16.0  ARI           1
 2025.0  17.0  ARI           1
 2025.0  18.0  ARI           4

Ja'Marr Chase (CIN), depth_team por semana 2025:
 season  week team  depth_team
 2025.0   1.0  CIN           1
 2025.0   2.0  CIN           1
 2025.0   3.0  CIN           1
 2025.0   4.0  CIN           1
 2025.0   5.0  CIN           1
 2025.0   6.0  CIN           1
 2025.0   7.0  CIN           1
 2025.0   8.0  CIN           1
 2025.0   9.0  CIN           1
 2025.0  11.0  CIN         

Ambos casos confirman `depth_team=1` (WR1 titular) en prácticamente todas las semanas jugadas
de 2025 -- coincide con lo que se sabe que pasó en la realidad. La única excepción (Harrison Jr.
en semana 18) es plausible: partido final de temporada sin impacto en playoffs para Arizona, donde
los equipos suelen rotar minutos -- no es un error del mapeo, es una fluctuación real del depth
chart esa semana específica.

In [6]:
cobertura = pd.DataFrame({
    "antes (solo ≤2024)": [
        data.cargar_depth_charts_historico([s]).shape[0] if s <= 2024 else 0
        for s in range(2016, 2026)
    ],
}, index=range(2016, 2026))

wr_stats = data.cargar_stats_semanales(range(2016, 2026))
wr_stats = wr_stats[wr_stats["position"] == "WR"]
cruce = wr_stats.merge(depth_unificado, on=["season", "week", "team", "player_id"], how="left")
cobertura_real = cruce.groupby("season")["depth_team"].apply(lambda s: s.notna().mean())
print("cobertura real de depth_team sobre filas WR reales, por temporada:")
print((cobertura_real * 100).round(1).astype(str) + "%")


cobertura real de depth_team sobre filas WR reales, por temporada:
season
2016    90.5%
2017    92.2%
2018    91.1%
2019    90.8%
2020    93.1%
2021    88.7%
2022    89.1%
2023    90.3%
2024    89.3%
2025    96.7%
Name: depth_team, dtype: object


Cobertura sube de **0%** a **~97%** para 2025 en adelante -- el ~9% de huecos en temporadas
≤2024 (sin cambios, ya existía) responde a la misma causa ya documentada en Fase 2: jugadores no
listados ese equipo/semana en la formación ofensiva (lesión, inactivo, etc.), no un error de
mapeo.

## 4. Impacto real en los modelos ya guardados

Se retoma exactamente el split y los hiperparámetros de `13_comparacion_modelos.ipynb`/
`18_validacion_temporada_actual.ipynb` (train 2016-2021, test 2024-2025) -- **sin buscar
hiperparámetros nuevos**, solo reentrenar con el `depth_team` corregido y comparar. Como el
esquema ≤2024 no cambió, el único año realmente afectado dentro del test es 2025 -- se reporta
el agregado 2024-2025 (comparable directo contra lo ya reportado) y también 2025 aislado (donde
vive el 100% del cambio real).

In [7]:
# Reconstruye exactamente la tabla vieja (depth_team roto en 2025+) para comparar de
# forma controlada -- no vive en features.py, es solo la version "antes" de este notebook.
def construir_antes(seasons):
    seasons = list(seasons)
    stats = data.cargar_stats_semanales(seasons)
    jugadores = data.cargar_jugadores()
    calendario = data.cargar_calendario(seasons)
    depth_hist = data.cargar_depth_charts_historico([s for s in seasons if s <= 2024])
    qb_titular = data.identificar_qb_titular(stats)
    wr = stats[stats["position"] == "WR"].copy()
    wr["es_prediccion"] = False

    wr = features.calcular_ratios_eficiencia(wr)
    wr["racr"] = (wr["receiving_yards"] / wr["receiving_air_yards"]).where(wr["receiving_air_yards"] > 0)
    wr = features.acotar_variable(wr, "racr", percentil=0.99)
    wr = features.agregar_promedios_jugador(wr, features.COLUMNAS_BASE_MODELADO, ventanas=(3, 5))
    wr = features.agregar_volatilidad_jugador(wr, ["receiving_yards", "targets"], ventana=5)
    wr = features.agregar_edad_experiencia(wr, jugadores)
    wr = features.agregar_experiencia_no_lineal(wr)
    wr = features.agregar_draft_pick(wr, jugadores)

    cal_home = calendario[["season", "week", "home_team", "home_rest", "div_game", "roof", "surface"]].rename(columns={"home_team": "team", "home_rest": "rest"})
    cal_away = calendario[["season", "week", "away_team", "away_rest", "div_game", "roof", "surface"]].rename(columns={"away_team": "team", "away_rest": "rest"})
    wr = wr.merge(pd.concat([cal_home, cal_away], ignore_index=True), on=["season", "week", "team"], how="left")

    depth_hist_r = depth_hist.rename(columns={"club_code": "team", "gsis_id": "player_id"})
    wr = wr.merge(depth_hist_r[["season", "week", "team", "player_id", "depth_team"]], on=["season", "week", "team", "player_id"], how="left")

    wr = features.agregar_ofensiva_equipo(wr, columnas=("receiving_yards", "targets"), ventanas=(3, 5))
    wr = features.agregar_cambio_qb_titular(wr, qb_titular)
    wr["cambio_qb_titular_num"] = wr["cambio_qb_titular"].astype("float")
    wr["cambio_equipo_num"] = wr["cambio_equipo"].astype("float")
    wr = features.agregar_interaccion(wr, "target_share_last5_avg", "equipo_receiving_yards_season_avg", "target_share_x_ofensiva_equipo")
    wr = features.agregar_interaccion(wr, "draft_pick", "anios_experiencia", "draft_pick_x_experiencia")
    wr = features.agregar_interaccion(wr, "cambio_qb_titular_num", "target_share_last5_avg", "cambio_qb_x_target_share")
    return wr

wr_antes = construir_antes(range(2016, 2026))
wr_despues = features.construir_tabla_modelado(range(2016, 2026))
print("tablas reconstruidas para comparacion controlada.")


tablas reconstruidas para comparacion controlada.


In [8]:
features_arbol = features.columnas_por_modelo("arbol")
HIPERPARAMETROS_YA_GUARDADOS = {
    "receptions": dict(max_depth=4, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1),
    "receiving_yards": dict(max_depth=3, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1),
    "receiving_tds": dict(objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1),
}

def comparar(wr, filtro_test):
    train_mask = wr["season"].between(2016, 2021)
    test_mask = filtro_test(wr)
    filas = {}
    for target, params in HIPERPARAMETROS_YA_GUARDADOS.items():
        modelo = XGBRegressor(**params)
        modelo.fit(wr.loc[train_mask, features_arbol], wr.loc[train_mask, target])
        pred = modelo.predict(wr.loc[test_mask, features_arbol])
        filas[target] = modeling.metricas(wr.loc[test_mask, target], pred)
    return pd.DataFrame(filas).T

test_2024_2025 = lambda wr: wr["season"].between(2024, 2025)
test_solo_2025 = lambda wr: wr["season"] == 2025

tabla_2024_2025 = pd.concat(
    {"antes": comparar(wr_antes, test_2024_2025), "despues": comparar(wr_despues, test_2024_2025)},
    axis=1,
)
tabla_2025 = pd.concat(
    {"antes": comparar(wr_antes, test_solo_2025), "despues": comparar(wr_despues, test_solo_2025)},
    axis=1,
)
print("test 2024-2025 (comparable directo contra 13_comparacion_modelos.ipynb):")
print(tabla_2024_2025.round(4).to_string())
print("\ntest solo 2025 (unico tramo realmente afectado por el fix):")
print(tabla_2025.round(4).to_string())


test 2024-2025 (comparable directo contra 13_comparacion_modelos.ipynb):
                   antes                   despues                 
                     mae     rmse      r2      mae     rmse      r2
receptions        1.3721   1.8397  0.4448   1.3647   1.8257  0.4532
receiving_yards  20.6469  28.0381  0.3630  20.6502  27.9639  0.3664
receiving_tds     0.2466   0.4775 -0.0454   0.2488   0.4765 -0.0410

test solo 2025 (unico tramo realmente afectado por el fix):
                   antes                   despues                 
                     mae     rmse      r2      mae     rmse      r2
receptions        1.3270   1.7816  0.4381   1.3124   1.7532  0.4560
receiving_yards  20.0237  27.3737  0.3605  20.0303  27.2237  0.3674
receiving_tds     0.2229   0.4561 -0.0365   0.2272   0.4540 -0.0270


In [9]:
mejora = pd.DataFrame({
    "mae_antes_2025": tabla_2025["antes"]["mae"],
    "mae_despues_2025": tabla_2025["despues"]["mae"],
})
mejora["mejora_mae_%"] = 100 * (mejora["mae_antes_2025"] - mejora["mae_despues_2025"]) / mejora["mae_antes_2025"]
mejora["r2_antes_2025"] = tabla_2025["antes"]["r2"]
mejora["r2_despues_2025"] = tabla_2025["despues"]["r2"]
print(mejora.round(4).to_string())


                 mae_antes_2025  mae_despues_2025  mejora_mae_%  r2_antes_2025  r2_despues_2025
receptions               1.3270            1.3124        1.0975         0.4381           0.4560
receiving_yards         20.0237           20.0303       -0.0328         0.3605           0.3674
receiving_tds            0.2229            0.2272       -1.9161        -0.0365          -0.0270


## 5. Lectura honesta del resultado

El efecto es **real pero modesto**, no un salto grande:

- **`receptions`**: mejora ~1.1% de MAE en 2025 (1.327 → 1.312), R² sube de 0.438 a 0.456.
- **`receiving_yards`**: MAE prácticamente sin cambio (20.02 → 20.03), pero R² sube de 0.360 a
  0.367 -- el fix aporta señal que R² capta mejor que MAE (más sensible a que se explique mejor
  la varianza, no solo al error absoluto promedio, que en este problema está dominado por el
  ruido semana a semana ya documentado desde Fase 3/4).
- **`receiving_tds`**: MAE empeora ligeramente (0.223 → 0.227, ~-1.9%), pero R² mejora (menos
  negativo: -0.036 → -0.027) -- mismo patrón de tensión MAE-vs-R² ya confirmado varias veces en
  este proyecto para este target específico (`14_receiving_tds_modelos_de_conteo.ipynb`,
  `20_mejora_del_modelo.ipynb`).

¿Por qué no es un cambio más grande, si `depth_team` es la 2ª variable más importante? Dos
razones concretas, no especulación: (1) el fix solo afecta filas de 2025 -- 2024 y años
anteriores usan exactamente el mismo dato que antes; (2) XGBoost ya maneja NaN de forma nativa
(aprende una dirección por default cuando falta el valor) y el modelo tiene otras variables
correlacionadas con la titularidad real (`target_share`, `wopr`, volumen reciente) que ya
compensaban parte de la ausencia de `depth_team`. El fix cierra una limitación de datos real y
documentada, con evidencia consistente de que ayuda (sube R² en los 3 targets), pero no era la
única señal de titularidad que el modelo tenía disponible.

**Decisión**: se adopta la unificación como parte del pipeline estándar (ya integrada en
`data.py`/`features.py` -- no es una rama opcional). Se reentrenan y regrardan los 3 modelos base
con el `depth_team` corregido, usando exactamente los mismos hiperparámetros ya elegidos (ningún
valor nuevo, ninguna búsqueda) -- es una corrección de datos, no una decisión de modelado.

In [10]:
import json, os

METRICAS_REFERENCIA = {
    "receptions": {"mae_test_2024_2025": round(tabla_2024_2025["despues"]["mae"]["receptions"], 4),
                   "r2_test_2024_2025": round(tabla_2024_2025["despues"]["r2"]["receptions"], 4),
                   "mae_legado_2025_real": 1.722, "mae_nuevo_2025_real": 1.644},
    "receiving_yards": {"mae_test_2024_2025": round(tabla_2024_2025["despues"]["mae"]["receiving_yards"], 4),
                         "r2_test_2024_2025": round(tabla_2024_2025["despues"]["r2"]["receiving_yards"], 4),
                         "mae_legado_2025_real": 27.049, "mae_nuevo_2025_real": 25.113},
    "receiving_tds": {"mae_test_2024_2025": round(tabla_2024_2025["despues"]["mae"]["receiving_tds"], 4),
                       "r2_test_2024_2025": round(tabla_2024_2025["despues"]["r2"]["receiving_tds"], 4),
                       "mae_legado_2025_real": 0.412, "mae_nuevo_2025_real": 0.341},
}

os.makedirs("../models", exist_ok=True)
for target, params in HIPERPARAMETROS_YA_GUARDADOS.items():
    modelo_final = XGBRegressor(**params)
    modelo_final.fit(wr_despues[features_arbol], wr_despues[target])
    modelo_final.save_model(f"../models/{target}_xgboost.json")

    metadata = {
        "target": target,
        "modelo": type(modelo_final).__name__,
        "hiperparametros": {k: v for k, v in modelo_final.get_params().items() if v is not None},
        "features_arbol_en_orden": features_arbol,
        "entrenado_con": "2016-2025 (toda la historia disponible al momento del guardado)",
        "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
        "metricas_referencia": METRICAS_REFERENCIA[target],
        "limitaciones_conocidas": [
            "depth_team cubre ~97% de las filas WR reales de 2025+ (antes 0%, ver 21_unificacion_depth_chart.ipynb); ~9% de nulos restantes en todo el historico son jugadores no listados esa semana en la formacion ofensiva, no un error de mapeo",
            "el modelo subestima sistematicamente las semanas boom (ver 13_comparacion_modelos.ipynb y 17_explicabilidad_shap.ipynb)",
        ],
    }
    with open(f"../models/{target}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

print("modelos base reguardados con depth_team unificado:", sorted(os.listdir("../models")))


modelos base reguardados con depth_team unificado: ['receiving_tds_metadata.json', 'receiving_tds_xgboost.json', 'receiving_yards_metadata.json', 'receiving_yards_xgboost.json', 'receptions_metadata.json', 'receptions_xgboost.json']


## Cierre

- `data.cargar_depth_charts_unificado()` resuelve la limitación de cobertura 0% en 2025+
  documentada desde Fase 2 -- integrada en `features.construir_tabla_modelado`, no es una rama
  aparte.
- Verificado con 2 casos reales conocidos (Harrison Jr., Chase) y con la cobertura real medida
  sobre filas WR reales (0% → ~97% en 2025).
- Impacto real, medido sin buscar hiperparámetros nuevos: mejora modesta pero consistente en R²
  para los 3 targets; MAE mejora en `receptions`, neutro en `receiving_yards`, ligeramente peor
  en `receiving_tds` (mismo patrón MAE-vs-R² ya conocido de ese target).
- Los 3 modelos base (`weekly/wr/models/`) quedan reentrenados con el dato corregido, mismos
  hiperparámetros que ya se habían elegido -- no una versión nueva del modelo, una versión
  corregida del dato que ya usaba.